In [1]:
from client import *
from common_imports import *

In [2]:
class DispatchPrompt(Executor):
    """Emit the same prompt downstream so fan-out edges can broadcast it."""

    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[str]) -> None:
        """Send one prompt message to all downstream expert branches."""
        await ctx.send_message(prompt)


In [3]:
@dataclass
class AggregatedInsights:
    """Typed container for consolidated expert perspectives."""

    research: str
    marketing: str
    legal: str

In [4]:
class AggregateInsights(Executor):
    """Join fan-in branch outputs and emit one consolidated report."""

    @handler
    async def aggregate(
        self,
        results: list[AgentExecutorResponse],
        ctx: WorkflowContext[Never, str],
    ) -> None:
        """Reduce a list of expert responses to one structured summary."""
        expert_outputs: dict[str, str] = {"research": "", "marketing": "", "legal": ""}
        # Process result.executor_id and result.agent_response.text
        for result in results:
            executor_id = result.executor_id.lower()
            text = result.agent_response.text
            if "research" in executor_id:
                expert_outputs["research"] = text
            elif "market" in executor_id:
                expert_outputs["marketing"] = text
            elif "legal" in executor_id:
                expert_outputs["legal"] = text

        aggregated = AggregatedInsights(
            research=expert_outputs["research"],
            marketing=expert_outputs["marketing"],
            legal=expert_outputs["legal"],
        )

        consolidated = (
            "=== Consolidated Launch Brief ===\n\n"
            f"Research Findings:\n{aggregated.research}\n\n"
            f"Marketing Angle:\n{aggregated.marketing}\n\n"
            f"Legal/Compliance Notes:\n{aggregated.legal}\n"
        )
        await ctx.yield_output(consolidated)


dispatcher = DispatchPrompt(id="dispatcher")

researcher = Agent(
    client=client,
    name="Researcher",
    instructions=(
        "You are an expert market researcher. "
        "Given the prompt, provide concise factual insights, opportunities, and risks. "
        "Use short bullet points."
    ),
)

marketer = Agent(
    client=client,
    name="Marketer",
    instructions=(
        "You are a marketing strategist. "
        "Given the prompt, propose clear value proposition and audience messaging. "
        "Use short bullet points."
    ),
)

legal = Agent(
    client=client,
    name="Legal",
    instructions=(
        "You are a legal and compliance reviewer. "
        "Given the prompt, list constraints, disclaimers, and policy concerns. "
        "Use short bullet points."
    ),
)

aggregator = AggregateInsights(id="aggregator")

workflow = (
    WorkflowBuilder(
        name="FanOutFanInEdges",
        description="Explicit fan-out/fan-in using edge groups.",
        start_executor=dispatcher,
        output_executors=[aggregator],
    )
    .add_fan_out_edges(dispatcher, [researcher, marketer, legal])
    .add_fan_in_edges([researcher, marketer, legal], aggregator)
    .build()
)



C:\Users\Ambarish Ganguly\AppData\Local\Temp\ipykernel_23520\3386397447.py:73: DeprecationWarning: `output_executors` is deprecated and will be removed in a future version; use `output_from` instead.
  WorkflowBuilder(


In [5]:
prompt = "We are launching a budget-friendly electric bike for urban commuters."

In [6]:
events = await workflow.run(prompt)
for output in events.get_outputs():
        print(output)

AgentExecutor 'Researcher': from_str handler invoked with an empty cache. If you are chaining from an AgentExecutor, the upstream custom executor may be emitting a plain str instead of using AgentExecutorResponse.with_text(...), which causes the full conversation context to be lost.
AgentExecutor 'Marketer': from_str handler invoked with an empty cache. If you are chaining from an AgentExecutor, the upstream custom executor may be emitting a plain str instead of using AgentExecutorResponse.with_text(...), which causes the full conversation context to be lost.
AgentExecutor 'Legal': from_str handler invoked with an empty cache. If you are chaining from an AgentExecutor, the upstream custom executor may be emitting a plain str instead of using AgentExecutorResponse.with_text(...), which causes the full conversation context to be lost.


=== Consolidated Launch Brief ===

Research Findings:
**Insights**  
- Growing demand for eco-friendly transportation in urban areas.  
- Rising fuel costs incentivize electric bike adoption.  
- Strong interest in affordable e-bikes among younger demographics and cost-conscious commuters.  
- Urban congestion drives demand for compact commuting solutions.  
- Governments offer subsidies for green mobility in many regions.  

**Opportunities**  
- Target first-time e-bike buyers with value pricing.  
- Focus on essential features like reliability, safety, and convenience.  
- Partner with local bike shops for distribution and servicing.  
- Work on fleet opportunities for delivery services or shared mobility.  
- Incorporate app-based features for navigation or tracking to add value.  

**Risks**  
- Intense competition from established and emerging brands.  
- High initial R&D and production costs for affordability.  
- Potential supply chain disruptions for batteries and components.  
- Customers may perceive "budget-friendly" as lower quality.  
- Regulatory barriers or certification in some regions could delay launch.

Marketing Angle:
**Value Proposition:**  
- Affordable eco-friendly transportation solution.  
- Designed for urban commuting with convenience and style.  
- Saves time and money compared to public transport and cars.  
- Low maintenance, high-efficiency electric mobility.

**Audience Messaging:**  
- "Commute Smarter, Not Harder: Meet the budget-friendly e-bike for urban explorers."  
- "Electric freedom at a price you can afford."  
- "Save on gas and public transport – go green without breaking the bank!"  
- "Your wallet-friendly solution to faster, eco-conscious city commuting."  
- "Effortless rides, everyday savings – perfect for city life."

Legal/Compliance Notes:
### Constraints:
- **Safety Compliance:** Must meet local and national electric bike safety regulations (e.g., speed limits, wattage
limits).
- **Battery and EMF Standards:** Ensure adherence to environmental and electromagnetic standards for battery safety
and disposal.
- **Licensing Laws:** Verify if usage requires registration, insurance, or a driver’s license.
- **Weight and Load Limits:** Adhere to guidelines for maximum load capacity and weight for urban usage.
- **Advertising Standards:** Avoid exaggerated claims about performance, range, and costs; ensure accuracy.
- **Consumer Protection Laws:** Transparent warranty, refund, and repair policies.

### Disclaimers:
- **Range and Battery Disclaimer:** Performance may vary based on terrain, rider weight, and usage.
- **Local Laws:** Riders should check local regulations regarding helmet use, bike lane access, and speed 
restrictions.
- **Maintenance Requirements:** Regular maintenance may be required to ensure performance and safety.
- **Risk Disclaimer:** Highlight potential risks, such as collisions or battery overheating, despite safety 
measures.

### Policy Concerns:
- **Environmental Impact:** Ensure responsible manufacturing and end-of-life battery recycling policies.
- **Data Privacy:** Comply with privacy regulations if the bike features app-based tracking or telematics.
- **Accessibility:** Address inclusivity concerns, such as affordability and usability for persons with 
disabilities.
- **Liability Exposure:** Establish clear terms to limit liability for misuse or improper maintenance.
- **Product Recall Preparedness:** Have a protocol for defects or safety issues that may require recalls.